In [ ]:
# 1. Paired t-test   # 2. Wilcoxon signed-rank test

In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, precision_recall_curve, average_precision_score
from scipy.stats import ttest_rel, wilcoxon
import os
import random
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ====================== 随机种子 ======================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

LOCAL_MODEL_DIR = "./hf_models"
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "roberta-base")

# ====================== 配置 checkpoint ======================
CLASS_CHECKPOINT = "best_Baseline.pt"  # 分类 baseline checkpoint
REG_CHECKPOINT = "best_dual_regression.pt"  # 回归 checkpoint

# ====================== 数据加载 ======================
df = pd.read_csv("mental_heath_unbanlanced.csv")
df = df.dropna(subset=["text", "status"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["status"])

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["label_encoded"].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df["label_encoded"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

# ====================== Dataset ======================
class AnalysisDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            "text": self.texts[idx]
        }
        return item

# ====================== 分类模型 ======================
class ClassifierModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, 4)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        return logits

# ====================== 回归模型 ======================
class DualRegressionModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        
        self.dep_head = nn.Linear(hidden, 1)
        self.sui_head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        dep_score = torch.sigmoid(self.dep_head(cls)).squeeze(-1)
        sui_score = torch.sigmoid(self.sui_head(cls)).squeeze(-1)
        return dep_score, sui_score

# ====================== 分数转分类（用于回归模型） ======================
def scores_to_class(dep_scores, sui_scores, dep_thresh=0.68, sui_thresh=0.72, anx_thresh=0.38):
    preds = []
    for d, s in zip(dep_scores, sui_scores):
        if s >= sui_thresh:
            preds.append(3)  # Suicidal
        elif d >= dep_thresh:
            preds.append(1)  # Depression
        elif d >= anx_thresh:
            preds.append(0)  # Anxiety
        else:
            preds.append(2)  # Normal
    return np.array(preds)

# ====================== 加载模型 ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

class_model = ClassifierModel(MODEL_PATH).to(DEVICE)
class_model.load_state_dict(torch.load(CLASS_CHECKPOINT, map_location=DEVICE))
class_model.eval()

reg_model = DualRegressionModel(MODEL_PATH).to(DEVICE)
reg_model.load_state_dict(torch.load(REG_CHECKPOINT, map_location=DEVICE))
reg_model.eval()

test_dataset = AnalysisDataset(test_texts, test_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ====================== 推理获取预测 ======================
print("正在推理测试集...")
all_class_preds = []
all_dep_scores = []
all_sui_scores = []
all_true_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].cpu().numpy()
        
        # 分类预测
        class_logits = class_model(input_ids, attention_mask)
        class_preds = torch.argmax(class_logits, dim=1).cpu().numpy()
        
        # 回归预测
        dep_score, sui_score = reg_model(input_ids, attention_mask)
        dep_score = dep_score.cpu().numpy()
        sui_score = sui_score.cpu().numpy()
        
        all_class_preds.extend(class_preds)
        all_dep_scores.extend(dep_score)
        all_sui_scores.extend(sui_score)
        all_true_labels.extend(labels)

all_class_preds = np.array(all_class_preds)
all_dep_scores = np.array(all_dep_scores)
all_sui_scores = np.array(all_sui_scores)
all_true_labels = np.array(all_true_labels)

# ====================== 回归转分类预测 ======================
reg_preds = scores_to_class(all_dep_scores, all_sui_scores)

# ====================== Bootstrap 采样生成多个指标样本 ======================
n_bootstraps = 100  # 采样次数（可调，100-1000）
boot_class_f1 = []
boot_reg_f1 = []

print("\n正在进行 Bootstrap 采样 (n={})...".format(n_bootstraps))
for _ in tqdm(range(n_bootstraps)):
    # 采样子集
    indices = np.random.choice(len(all_true_labels), len(all_true_labels), replace=True)
    
    true_sub = all_true_labels[indices]
    class_sub = all_class_preds[indices]
    reg_sub = reg_preds[indices]
    
    # 计算加权 F1
    boot_class_f1.append(f1_score(true_sub, class_sub, average='weighted'))
    boot_reg_f1.append(f1_score(true_sub, reg_sub, average='weighted'))

boot_class_f1 = np.array(boot_class_f1)
boot_reg_f1 = np.array(boot_reg_f1)

# ====================== 统计检验 ======================
# 1. Paired t-test
t_stat, p_val_t = ttest_rel(boot_reg_f1, boot_class_f1)

# 2. Wilcoxon signed-rank test (非参数版)
stat_w, p_val_w = wilcoxon(boot_reg_f1 - boot_class_f1)

print("\n=== 统计检验结果 (Regression vs Classification Weighted F1) ===")
print(f"回归均值 F1: {np.mean(boot_reg_f1):.4f} (std: {np.std(boot_reg_f1):.4f})")
print(f"分类均值 F1: {np.mean(boot_class_f1):.4f} (std: {np.std(boot_class_f1):.4f})")
print(f"Paired t-test: t-stat = {t_stat:.4f}, p-value = {p_val_t:.4e}")
print(f"Wilcoxon test: stat = {stat_w:.4f}, p-value = {p_val_w:.4e}")

if p_val_t < 0.05:
    print("Paired t-test: 改进统计显著 (p<0.05)")
else:
    print("Paired t-test: 改进不统计显著 (p>=0.05)")

if p_val_w < 0.05:
    print("Wilcoxon: 改进统计显著 (p<0.05)")
else:
    print("Wilcoxon: 改进不统计显著 (p>=0.05)")

print("\n实验完成！如果 p-value < 0.05，证明回归改进显著。")

Using device: cuda


Loading weights: 100%|██| 197/197 [00:00<00:00, 506.13it/s, Materializing param=encoder.layer.11.output.dense.weight]
RobertaModel LOAD REPORT from: ./hf_models\roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██| 197/197 [00:00<00:00, 556.52it/s, Materializing param=encoder.layer.11.o

正在推理测试集...


100%|██████████████████████████████████████████████████████████████████████████████| 233/233 [06:25<00:00,  1.66s/it]



正在进行 Bootstrap 采样 (n=100)...


100%|█████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 268.82it/s]


=== 统计检验结果 (Regression vs Classification Weighted F1) ===
回归均值 F1: 0.8048 (std: 0.0050)
分类均值 F1: 0.8527 (std: 0.0043)
Paired t-test: t-stat = -105.9539, p-value = 1.0316e-103
Wilcoxon test: stat = 0.0000, p-value = 3.8966e-18
Paired t-test: 改进统计显著 (p<0.05)
Wilcoxon: 改进统计显著 (p<0.05)

实验完成！如果 p-value < 0.05，证明回归改进显著。
